<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_08_practicum_big_o/lab_lesson_08_taxi_big_o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторія: таксі і Big O

**Урок 8 · Практикум П1.** Диспетчерська служба таксі починала з тисячі поїздок на день. Місто росте. Для кожної задачі диспетчера в нас є два **правильні** рішення — вони завжди дають однакову відповідь. Питання лабораторії: **що станеться з кожним рішенням, коли поїздок стане вдвічі, в десять, у тисячу разів більше?**

Кожен дослід іде за однією схемою:

1. **Передбач** — впиши свій прогноз, нічого не запускаючи.
2. **Запусти** — перевір, що рішення дають однакову відповідь, і порахуй кроки для n = 500, 1000, 2000, 4000.
3. **Поясни** — порівняй прогноз з виміряним і знайди в коді, звідки взялася різниця.

Головний інструмент — **таблиця подвоєння**: n щоразу вдвічі більше, і ми дивимось, у скільки разів зросла кількість кроків. ×2 означає, що робота росте разом з n (`O(n)`); ×4 — що робота росте як n² (`O(n²)`).

Теорія — у книзі: [Урок 8. П1. Big O + базові задачі](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_08/).

## Розминка. Чому не секунди?

Здавалося б, найпростіше — заміряти час. Запустимо ту саму функцію п'ять разів на тих самих даних.

In [ ]:
import time


def total_fare(fares):
    total = 0
    for fare in fares:
        total += fare
    return total


fares = [150.0] * 200_000
for attempt in range(1, 6):
    start = time.perf_counter()
    total_fare(fares)
    elapsed = time.perf_counter() - start
    print(f"Запуск {attempt}: {elapsed * 1000:.2f} мс")

Той самий код, ті самі дані — а мілісекунди щоразу трохи інші. На іншому комп'ютері вони будуть зовсім інші. Час залежить від заліза й від того, що ще робить система в цю мить.

Тому ми **рахуємо кроки**: одне порівняння двох значень або одна перевірка в множині чи словнику — один крок. Кроки однакові на будь-якому комп'ютері. Кожне рішення в лабораторії саме рахує свої кроки й повертає пару `(відповідь, кроки)`.

## Інструменти

Клітинку нижче достатньо просто запустити. У ній:

- `Trip` і `make_trips(n)` — n поїздок (номер, водій, клієнт, район, вартість). Дані генеруються з фіксованим `seed`, тому однакові на кожному запуску;
- `make_shifts(n)` — водії понеділка й вівторка, приблизно 20 % працювали в обидва дні;
- `doubling_table(...)` і `print_doubling_table(...)` — таблиця подвоєння;
- `plot_growth(...)` — графік кроків обох рішень;
- `check_prediction(...)` — порівнює твій прогноз з виміряним.

In [ ]:
import random
import time
from typing import NamedTuple

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

ZONES = ['Поділ', 'Оболонь', 'Печерськ', "Лук'янівка", 'Троєщина', 'Позняки', 'Виноградар', 'Теремки', 'Святошин', 'Голосіїв', 'Дарниця', 'Нивки']


class Trip(NamedTuple):
    trip_id: int
    driver: str
    client: str
    zone: str
    fare: float


def make_trips(n, seed=8):
    """Список з n поїздок. Номери поїздок унікальні, клієнти повторюються."""
    rng = random.Random(seed)
    trip_ids = rng.sample(range(100_000, 100_000 + 10 * n), n)
    drivers = max(1, n // 10)
    clients = max(1, n // 2)
    return [
        Trip(
            trip_id=trip_ids[i],
            driver=f"D-{rng.randrange(drivers):05d}",
            client=f"C-{rng.randrange(clients):06d}",
            zone=rng.choice(ZONES),
            fare=round(rng.uniform(80, 600), 2),
        )
        for i in range(n)
    ]


def make_shifts(n, seed=8):
    """Два списки по n різних водіїв (понеділок, вівторок); спільних — приблизно 20 %."""
    rng = random.Random(seed)
    pool = [f"D-{i:05d}" for i in range(2 * n)]
    monday = rng.sample(pool, n)
    common = rng.sample(monday, n // 5)
    monday_set = set(monday)
    others = [d for d in pool if d not in monday_set]
    tuesday = common + rng.sample(others, n - len(common))
    rng.shuffle(tuesday)
    return monday, tuesday


def count_steps(func, make_input, sizes):
    """Для кожного n: (n, кроки). Дані для кожного n генеруються наново."""
    rows = []
    for n in sizes:
        _, steps = func(*make_input(n))
        rows.append((n, steps))
    return rows


def doubling_table(func, make_input, sizes):
    """Рядки (n, кроки, у скільки разів більше, ніж для попереднього n).

    Якщо n щоразу подвоюється, відношення ≈ 2 означає O(n), ≈ 4 — O(n²).
    """
    rows = []
    previous = None
    for n, steps in count_steps(func, make_input, sizes):
        ratio = None if previous is None else round(steps / previous, 2)
        rows.append((n, steps, ratio))
        previous = steps
    return rows


def print_doubling_table(rows):
    print(f"{'n':>8} {'кроки':>14} {'× до попереднього':>18}")
    for n, steps, ratio in rows:
        shown = "—" if ratio is None else f"×{ratio}"
        print(f"{n:>8} {steps:>14,} {shown:>18}")


def seconds(func, *args, repeat=3):
    """Найкращий із кількох замірів часу, у секундах."""
    best = None
    for _ in range(repeat):
        start = time.perf_counter()
        func(*args)
        elapsed = time.perf_counter() - start
        if best is None or elapsed < best:
            best = elapsed
    return best


def human_time(total_seconds):
    """Секунди → зрозумілий рядок: мс, с, хв, год, дні, роки."""
    units = [
        (365 * 24 * 3600, "р."), (24 * 3600, "дн."), (3600, "год"),
        (60, "хв"), (1, "с"),
    ]
    if total_seconds < 1:
        return f"{total_seconds * 1000:.1f} мс"
    for size, name in units:
        if total_seconds >= size:
            return f"{total_seconds / size:.1f} {name}"
    return f"{total_seconds:.1f} с"


SLOW_COLOR = "#eb6834"
FAST_COLOR = "#2a78d6"


def plot_growth(slow, fast, make_input, sizes, title):
    """Графік: скільки кроків роблять обидва рішення для кожного n."""
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for func, name, color in ((slow, "повільне", SLOW_COLOR), (fast, "швидке", FAST_COLOR)):
        rows = count_steps(func, make_input, sizes)
        ns = [n for n, _ in rows]
        steps = [s for _, s in rows]
        ax.plot(ns, steps, marker="o", markersize=7, linewidth=2, color=color)
        ax.annotate(name, (ns[-1], steps[-1]), xytext=(8, 0), textcoords="offset points",
                    va="center", color="#52514e")
    ax.set_title(title, loc="left")
    ax.set_xlabel("кількість поїздок n")
    ax.set_ylabel("кроків")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " ")))
    ax.grid(axis="y", color="#ecebe8")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


def check_prediction(name, guess, rows):
    """Порівнює прогноз (1, 2, 4 або 8) з останнім виміряним відношенням."""
    measured = rows[-1][2]
    nearest = min((1, 2, 4, 8), key=lambda option: abs(option - measured))
    if guess is None:
        print(f"{name}: спершу впиши свій прогноз, потім запусти клітинку ще раз.")
    elif guess == nearest:
        print(f"✅ {name}: прогноз ×{guess} підтвердився (виміряно ×{measured}).")
    else:
        print(f"❌ {name}: прогноз ×{guess}, а виміряно ×{measured}. Чому? Подивись на цикли в коді.")


print("Інструменти готові. Приклад поїздки:", make_trips(1)[0])

## Дослід 1. Повторний номер поїздки

Бухгалтерія помітила, що одна поїздка могла потрапити в журнал двічі. Треба перевірити: **чи є в журналі дві поїздки з однаковим номером?**

**Передбач до запуску.** Поїздок стало **вдвічі більше**. У скільки разів зросте кількість кроків кожного рішення: 1, 2, 4 чи 8? Впиши відповіді в клітинку нижче.

In [ ]:
GUESS_SLOW_1 = None   # впиши 1, 2, 4 або 8
GUESS_FAST_1 = None

In [ ]:
def has_duplicate_ids_slow(ids):
    """Порівнює кожну пару номерів: O(n²)."""
    steps = 0
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            steps += 1
            if ids[i] == ids[j]:
                return True, steps
    return False, steps


def has_duplicate_ids_fast(ids):
    """Один прохід і множина вже побачених номерів: O(n)."""
    steps = 0
    seen = set()
    for trip_id in ids:
        steps += 1
        if trip_id in seen:
            return True, steps
        seen.add(trip_id)
    return False, steps


def input_ids(n):
    return ([t.trip_id for t in make_trips(n)],)

Запускаємо: спершу перевіряємо, що рішення **однаково правильні**, потім рахуємо кроки для n = 500, 1000, 2000, 4000.

In [ ]:
SIZES = [500, 1000, 2000, 4000]   # щоразу вдвічі більше

for n in (1, 10, 300):
    assert has_duplicate_ids_slow(*input_ids(n))[0] == has_duplicate_ids_fast(*input_ids(n))[0]
print("Відповіді однакові — обидва рішення правильні.\n")

slow_rows = doubling_table(has_duplicate_ids_slow, input_ids, SIZES)
fast_rows = doubling_table(has_duplicate_ids_fast, input_ids, SIZES)
print("Повільне:")
print_doubling_table(slow_rows)
print("\nШвидке:")
print_doubling_table(fast_rows)
print()
check_prediction("Повільне", GUESS_SLOW_1, slow_rows)
check_prediction("Швидке", GUESS_FAST_1, fast_rows)
plot_growth(has_duplicate_ids_slow, has_duplicate_ids_fast, input_ids, SIZES, "Повторний номер поїздки")

**Чому так.** Повільне рішення порівнює кожну пару номерів: для n поїздок це `n·(n−1)/2` порівнянь. Подвоїли n — пар стало майже вчетверо більше. Швидке дивиться на кожен номер **один раз**, а перевірка `trip_id in seen` для множини — один крок.

Журнал без дублікатів — **найгірший випадок**: щоб сказати «повторів немає», повільне рішення мусить перевірити всі пари. Big O описує саме найгірший випадок — це гарантія «не повільніше, ніж».

Ціна швидкості — пам'ять: множина `seen` зберігає до n номерів.

### Пастка: «я вдвічі зменшив роботу — отже, тепер O(n)?»

У повільному рішенні внутрішній цикл починається з `i + 1`, а не з нуля: кожну пару порівнюємо один раз, а не двічі. Це справді вдвічі менше роботи. Але чи змінився клас складності? Порівняй з версією, яка перевіряє всі пари двічі.

In [ ]:
def has_duplicate_ids_all_pairs(ids):
    steps = 0
    for i in range(len(ids)):
        for j in range(len(ids)):
            if i != j:
                steps += 1
                if ids[i] == ids[j]:
                    return True, steps
    return False, steps


print_doubling_table(doubling_table(has_duplicate_ids_all_pairs, input_ids, SIZES))

Версія «всі пари двічі» робить удвічі більше кроків, але відношення при подвоєнні те саме — ×4. Множник 2 чи ½ не змінює того, **як** росте робота. Тому в Big O константи відкидають: `n·(n−1)/2` — це `O(n²)`.

## Дослід 2. Водії двох змін

Диспетчер хоче привітати водіїв, які працювали **і в понеділок, і у вівторок**. Є два списки водіїв — по одному на кожен день.

У повільному рішенні замість `if driver in tuesday:` написано внутрішній цикл з лічильником. Він робить **те саме**, що `in` для списку: переглядає вівторок від початку, поки не знайде водія.

**Передбач до запуску.** Поїздок стало **вдвічі більше**. У скільки разів зросте кількість кроків кожного рішення: 1, 2, 4 чи 8? Впиши відповіді в клітинку нижче.

In [ ]:
GUESS_SLOW_2 = None   # впиши 1, 2, 4 або 8
GUESS_FAST_2 = None

In [ ]:
def common_drivers_slow(monday, tuesday):
    """`driver in tuesday` для списку — прихований цикл: O(n·m).

    Цикл по tuesday нижче робить те саме, що `in` для списку,
    лише з лічильником, щоб прихована робота стала видимою.
    """
    steps = 0
    result = []
    for driver in monday:
        for other in tuesday:
            steps += 1
            if other == driver:
                result.append(driver)
                break
    return result, steps


def common_drivers_fast(monday, tuesday):
    """Множина вівторка будується один раз, далі кожна перевірка — один крок: O(n + m)."""
    steps = 0
    tuesday_set = set()
    for driver in tuesday:
        steps += 1
        tuesday_set.add(driver)
    result = []
    for driver in monday:
        steps += 1
        if driver in tuesday_set:
            result.append(driver)
    return result, steps


def input_shifts(n):
    return make_shifts(n)

Запускаємо: спершу перевіряємо, що рішення **однаково правильні**, потім рахуємо кроки для n = 500, 1000, 2000, 4000.

In [ ]:
SIZES = [500, 1000, 2000, 4000]   # щоразу вдвічі більше

for n in (1, 10, 300):
    assert common_drivers_slow(*input_shifts(n))[0] == common_drivers_fast(*input_shifts(n))[0]
print("Відповіді однакові — обидва рішення правильні.\n")

slow_rows = doubling_table(common_drivers_slow, input_shifts, SIZES)
fast_rows = doubling_table(common_drivers_fast, input_shifts, SIZES)
print("Повільне:")
print_doubling_table(slow_rows)
print("\nШвидке:")
print_doubling_table(fast_rows)
print()
check_prediction("Повільне", GUESS_SLOW_2, slow_rows)
check_prediction("Швидке", GUESS_FAST_2, fast_rows)
plot_growth(common_drivers_slow, common_drivers_fast, input_shifts, SIZES, "Водії двох змін")

**Чому так.** Для кожного з n водіїв понеділка повільне рішення переглядає до m водіїв вівторка: цикл у циклі, `n·m` кроків. Швидке будує множину вівторка **один раз** (m кроків), а далі кожна перевірка — один крок: разом `n + m`.

Два **послідовні** цикли додаються: `O(n + m)`. Два **вкладені** — множаться: `O(n·m)`.

### Прихований цикл: `in` для списку

Запис `if driver in tuesday:` виглядає як один крок. Заміряймо час `in` для списку й для множини на однакових даних: шукаємо водія, якого немає (найгірший випадок).

In [ ]:
monday, tuesday = make_shifts(20_000)
tuesday_set = set(tuesday)
missing = "D-99999"

list_time = seconds(lambda: missing in tuesday)
set_time = seconds(lambda: missing in tuesday_set)
print(f"in для списку:   {list_time * 1_000_000:9.1f} мкс")
print(f"in для множини:  {set_time * 1_000_000:9.1f} мкс")
print(f"Список повільніший приблизно в {list_time / set_time:,.0f} раз")

`in` для списку — це цикл, захований у два символи: Python переглядає елементи по черзі. `in` для множини чи словника — один крок завдяки хешуванню (урок 16). Тому `in` для списку всередині `for` — найчастіша прихована `O(n²)`.

## Дослід 3. Клієнти для розсилки

Відділ маркетингу хоче надіслати листи клієнтам — кожному **один раз**, у порядку їхньої першої поїздки. У журналі клієнти повторюються.

**Передбач до запуску.** Поїздок стало **вдвічі більше**. У скільки разів зросте кількість кроків кожного рішення: 1, 2, 4 чи 8? Впиши відповіді в клітинку нижче.

In [ ]:
GUESS_SLOW_3 = None   # впиши 1, 2, 4 або 8
GUESS_FAST_3 = None

In [ ]:
def unique_clients_slow(clients):
    """`client not in result` для списку, що росте: O(n²)."""
    steps = 0
    result = []
    for client in clients:
        found = False
        for known in result:
            steps += 1
            if known == client:
                found = True
                break
        if not found:
            result.append(client)
    return result, steps


def unique_clients_fast(clients):
    """Допоміжна множина для перевірки і список для порядку: O(n)."""
    steps = 0
    result = []
    seen = set()
    for client in clients:
        steps += 1
        if client not in seen:
            seen.add(client)
            result.append(client)
    return result, steps


def input_clients(n):
    return ([t.client for t in make_trips(n)],)

Запускаємо: спершу перевіряємо, що рішення **однаково правильні**, потім рахуємо кроки для n = 500, 1000, 2000, 4000.

In [ ]:
SIZES = [500, 1000, 2000, 4000]   # щоразу вдвічі більше

for n in (1, 10, 300):
    assert unique_clients_slow(*input_clients(n))[0] == unique_clients_fast(*input_clients(n))[0]
print("Відповіді однакові — обидва рішення правильні.\n")

slow_rows = doubling_table(unique_clients_slow, input_clients, SIZES)
fast_rows = doubling_table(unique_clients_fast, input_clients, SIZES)
print("Повільне:")
print_doubling_table(slow_rows)
print("\nШвидке:")
print_doubling_table(fast_rows)
print()
check_prediction("Повільне", GUESS_SLOW_3, slow_rows)
check_prediction("Швидке", GUESS_FAST_3, fast_rows)
plot_growth(unique_clients_slow, unique_clients_fast, input_clients, SIZES, "Клієнти для розсилки")

**Чому так.** `client not in result` перевіряє **список, що росте**: що більше клієнтів уже знайдено, то довша кожна наступна перевірка. Сума таких перевірок — `O(n²)`, хоча в коді лише один видимий `for`.

Швидке рішення розділяє ролі: множина `seen` відповідає «чи був уже» за один крок, а список `result` лише тримає порядок.

### Пастка: `list(set(...))`

Найкоротший спосіб прибрати повтори — `list(set(clients))`. Це `O(n)`, але чи зберігається порядок першої поїздки?

In [ ]:
clients = input_clients(20)[0]
ordered, _ = unique_clients_fast(clients)
print("Порядок першої поїздки:", ordered[:5])
print("list(set(...)):         ", list(set(clients))[:5])
print("Однаковий порядок?", ordered == list(set(clients)))

Множина не зберігає порядок вставки, тому `list(set(...))` підходить, лише коли порядок не важливий. Коли важливий — множина для перевірки плюс список для порядку, як у швидкому рішенні.

## Дослід 4. Найшвидше повернення

Аналітик питає: **через скільки поїздок найшвидше повернувся хтось із клієнтів?** Потрібна найменша відстань між двома поїздками одного клієнта в журналі (−1, якщо повторів немає).

**Передбач до запуску.** Поїздок стало **вдвічі більше**. У скільки разів зросте кількість кроків кожного рішення: 1, 2, 4 чи 8? Впиши відповіді в клітинку нижче.

In [ ]:
GUESS_SLOW_4 = None   # впиши 1, 2, 4 або 8
GUESS_FAST_4 = None

In [ ]:
def nearest_repeat_slow(clients):
    """Перебирає всі пари поїздок: O(n²). Повертає -1, якщо повторів немає."""
    steps = 0
    best = -1
    for i in range(len(clients)):
        for j in range(i + 1, len(clients)):
            steps += 1
            if clients[i] == clients[j] and (best == -1 or j - i < best):
                best = j - i
    return best, steps


def nearest_repeat_fast(clients):
    """Словник «клієнт → остання поїздка»: один прохід, O(n)."""
    steps = 0
    best = -1
    last_seen = {}
    for i, client in enumerate(clients):
        steps += 1
        if client in last_seen and (best == -1 or i - last_seen[client] < best):
            best = i - last_seen[client]
        last_seen[client] = i
    return best, steps


def input_clients(n):
    return ([t.client for t in make_trips(n)],)

Запускаємо: спершу перевіряємо, що рішення **однаково правильні**, потім рахуємо кроки для n = 500, 1000, 2000, 4000.

In [ ]:
SIZES = [500, 1000, 2000, 4000]   # щоразу вдвічі більше

for n in (1, 10, 300):
    assert nearest_repeat_slow(*input_clients(n))[0] == nearest_repeat_fast(*input_clients(n))[0]
print("Відповіді однакові — обидва рішення правильні.\n")

slow_rows = doubling_table(nearest_repeat_slow, input_clients, SIZES)
fast_rows = doubling_table(nearest_repeat_fast, input_clients, SIZES)
print("Повільне:")
print_doubling_table(slow_rows)
print("\nШвидке:")
print_doubling_table(fast_rows)
print()
check_prediction("Повільне", GUESS_SLOW_4, slow_rows)
check_prediction("Швидке", GUESS_FAST_4, fast_rows)
plot_growth(nearest_repeat_slow, nearest_repeat_fast, input_clients, SIZES, "Найшвидше повернення")

**Чому так.** Повільне рішення перебирає всі `n·(n−1)/2` пар поїздок. Швидке використовує спостереження: найближчий повтор клієнта — це його **попередня** поїздка. Тому досить пам'ятати у словнику, де кожен клієнт їздив востаннє: один прохід, `O(n)`.

Ціна — пам'ять під словник: по запису на кожного клієнта.

### Пастка: «додам break — і стане швидше»

Здається, що можна зупинитися, щойно знайшли перший повтор. Але найкоротший може бути далі в журналі: перший знайдений повтор не обов'язково найближчий.

In [ ]:
journal = ["C-1", "C-2", "C-3", "C-1", "C-4", "C-4"]
print("Правильна відповідь:", nearest_repeat_fast(journal)[0])

for i in range(len(journal)):
    for j in range(i + 1, len(journal)):
        if journal[i] == journal[j]:
            print("Перший знайдений повтор:", j - i, f"(клієнт {journal[i]})")
            break
    else:
        continue
    break

Ранній вихід пришвидшує окремі «вдалі» журнали, але дає **неправильну** відповідь, а в найгіршому випадку (повторів немає) повільне рішення однаково мусить перевірити всі пари. Big O оцінює найгірший випадок, і `break` його не змінює.

## Місто росте

Заміряємо, скільки часу кожне рішення витрачає на 2000 поїздок, і оцінюємо час для міста з мільйоном поїздок: для `O(n)` множимо на відношення розмірів, для `O(n²)` — на його квадрат.

**Передбач:** скільки чекатиме повільне рішення на 1 000 000 поїздок — секунди, хвилини, години чи дні?

In [ ]:
n = 2000
target = 1_000_000
scale = target / n
pairs = [
    ("Повторний номер", has_duplicate_ids_slow, has_duplicate_ids_fast, input_ids),
    ("Водії двох змін", common_drivers_slow, common_drivers_fast, input_shifts),
    ("Клієнти для розсилки", unique_clients_slow, unique_clients_fast, input_clients),
    ("Найшвидше повернення", nearest_repeat_slow, nearest_repeat_fast, input_clients),
]

print(f"{'задача':<22} {'повільне O(n²)':>16} {'швидке O(n)':>14}")
for title, slow, fast, make_input in pairs:
    args = make_input(n)
    slow_time = seconds(slow, *args, repeat=1) * scale ** 2
    fast_time = seconds(fast, *args) * scale
    print(f"{title:<22} {human_time(slow_time):>16} {human_time(fast_time):>14}")

Точні числа залежать від комп'ютера, але картина однакова скрізь: швидкі рішення впораються за частки секунди, повільні — за години або дні. На 2000 поїздок обидва рішення працювали «миттєво», і різниці ніхто б не помітив. Саме тому складність оцінюють **до** того, як даних стане багато.

Висновок лабораторії:

| Шаблон у коді | Як росте робота |
|---|---|
| один прохід, перевірки в `set` / `dict` | `O(n)` — ×2 при подвоєнні n |
| цикл у циклі по тих самих даних | `O(n²)` — ×4 при подвоєнні n |
| `in` для списку всередині циклу | прихована `O(n²)` |
| множник ½ чи 2 перед n² | не змінює класу: однаково `O(n²)` |

## Самостійне завдання: студенти двох курсів

Інший контекст, та сама структура. Є списки студентів курсу Python і курсу SQL (імена унікальні в межах курсу). Напиши дві функції, що повертають пару `(студенти обох курсів у порядку списку Python, кроки)`:

- `both_courses_slow(python, sql)` — з вкладеним циклом по `sql`;
- `both_courses_fast(python, sql)` — з множиною.

Потім перевір, що відповіді однакові, і побудуй таблиці подвоєння. **Передбач** відношення до запуску.

In [ ]:
def make_courses(n, seed=8):
    rng = random.Random(seed)
    pool = [f"S-{i:05d}" for i in range(2 * n)]
    python = rng.sample(pool, n)
    sql = rng.sample(pool, n)
    return python, sql


# YOUR CODE HERE
# BEGIN SOLUTION
def both_courses_slow(python, sql):
    steps = 0
    result = []
    for student in python:
        for other in sql:
            steps += 1
            if other == student:
                result.append(student)
                break
    return result, steps


def both_courses_fast(python, sql):
    steps = 0
    sql_set = set()
    for student in sql:
        steps += 1
        sql_set.add(student)
    result = []
    for student in python:
        steps += 1
        if student in sql_set:
            result.append(student)
    return result, steps
# END SOLUTION


assert both_courses_slow(["a", "b", "c"], ["c", "a"])[0] == ["a", "c"]
assert both_courses_fast(["a", "b", "c"], ["c", "a"])[0] == ["a", "c"]
for n in (1, 50, 400):
    assert both_courses_slow(*make_courses(n))[0] == both_courses_fast(*make_courses(n))[0]
print_doubling_table(doubling_table(both_courses_slow, make_courses, SIZES))
print()
print_doubling_table(doubling_table(both_courses_fast, make_courses, SIZES))

## Самоперевірка

1. Два рішення завжди дають однакову відповідь. Чи означає це, що вони однаково ефективні?
2. При подвоєнні n кількість кроків зросла приблизно ×4. Який це клас складності?
3. Чому `if x in my_list:` усередині `for` часто перетворює `O(n)` на `O(n²)`?
4. Внутрішній цикл починається з `i + 1` замість 0. Чи змінився клас складності?
5. Чим платять швидкі рішення цієї лабораторії за швидкість?

<details>
<summary>Відповіді</summary>

1. Ні. Правильність і ефективність — різні питання: Big O не перевіряє, чи правильна відповідь, а описує, як росте робота.
2. `O(n²)`: робота росте як квадрат розміру даних.
3. `in` для списку переглядає елементи по черзі — це прихований цикл. Цикл у циклі дає `n·m` кроків.
4. Ні: кроків удвічі менше, але вони ростуть так само — ×4 при подвоєнні n. Константи в Big O відкидають.
5. Пам'яттю: множина чи словник зберігає до n елементів.

</details>

## Далі

- Необов'язково: той самий дослід в інтерактивному застосунку — [`taxi_lab/`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/tree/main/module_1/lessons/lesson_08_practicum_big_o/taxi_lab) (Streamlit, запуск на своєму комп'ютері).
- Ноутбук заняття з базовими задачами практикуму — FizzBuzz, паліндром, шифр Цезаря: [`note_lesson_08_big_o.ipynb`](https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_08_practicum_big_o/note_lesson_08_big_o.ipynb).
- Практикум П2 (урок 11, «Пошук») додасть ще один клас: `O(log n)` — коли дані відсортовані, половину можна відкидати на кожному кроці.